# cloudynight Dataset - DAG Cloud Detection Pipeline

**Kaynak**: Mommert (2020), *Cloud Identification from All-sky Camera Data with Machine Learning*
DOI: [10.3847/1538-3881/ab744f](https://doi.org/10.3847/1538-3881/ab744f)

| Ozellik | Deger |
|---------|-------|
| Toplam goruntu | 20 adet FITS gece goruntusu |
| Format | .fits.bz2 (FITS + bzip2) |
| Etiket | y_train.dat (33 alt bolge) |
| Kamera | All-sky (Lowell Gozlemevi, ABD) |

Bu notebook DAG pipeline kodunun FITS formatiyla uyumlu calistigini dogrular.

## 1. Kutuphaneler

In [ ]:
import sys
sys.path.append('..')

import os
import bz2
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import warnings
warnings.filterwarnings('ignore')

from dag_cld import env
from dag_cld import ast
from dag_cld import mask
from dag_cld import teacher

print('Kutuphaneler yuklendi!')

## 2. Moduller Baslat

In [ ]:
logger = env.Logger(blabla=False)
fop    = env.File(logger)
ima    = ast.Image(logger)
pmask  = mask.Polar(logger)

svm_clf = teacher.SVM(logger)
knn_clf = teacher.KNN(logger)
lr_clf  = teacher.LR(logger)
nb_clf  = teacher.NB(logger)
cnn_clf = teacher.CNN(logger)

mask_coordinates = {
    'E':  ((20, 70), (45, 135)),
    'S':  ((20, 70), (135, 225)),
    'W':  ((20, 70), (225, 315)),
    'N':  ((20, 70), (315, 405)),
    'ZE': ((70, 90), (45, 135)),
    'ZS': ((70, 90), (135, 225)),
    'ZW': ((70, 90), (225, 315)),
    'ZN': ((70, 90), (315, 405)),
}

print('Moduller baslatildi!')
print(f'GokYuzu bolgeleri: {list(mask_coordinates.keys())}')

## 3. Etiketleri Oku (y_train.dat)

In [ ]:
IMAGES_DIR  = '../cloudynight_data/example_data/images'
YTRAIN_FILE = os.path.join(IMAGES_DIR, 'y_train.dat')

def load_labels(filepath):
    labels = {}
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            img_id = parts[0]
            flags  = [int(x) for x in parts[1:]]
            cloud_ratio = sum(flags) / len(flags)
            label = 0 if cloud_ratio > 0.5 else 1
            labels[img_id] = label
    return labels

labels = load_labels(YTRAIN_FILE)

cloudy_count = sum(1 for v in labels.values() if v == 0)
clear_count  = sum(1 for v in labels.values() if v == 1)
print(f'Toplam: {len(labels)} goruntu')
print(f'Bulutlu: {cloudy_count} | Acik: {clear_count}')
for img_id, label in labels.items():
    status = 'BULUTLU' if label == 0 else 'ACIK'
    print(f'  Goruntu {img_id}: {status}')

## 4. FITS Goruntu Okuma

In [ ]:
def read_fits_bz2(filepath):
    with bz2.open(filepath, 'rb') as f:
        with fits.open(f) as hdul:
            data = hdul[0].data
            if data is None and len(hdul) > 1:
                data = hdul[1].data
            return data.astype(np.float64)

test_file = os.path.join(IMAGES_DIR, '000.fits.bz2')
test_data = read_fits_bz2(test_file)

print('Goruntu okundu!')
print(f'  Boyut  : {test_data.shape}')
print(f'  Min    : {test_data.min():.1f}')
print(f'  Max    : {test_data.max():.1f}')
print(f'  Dtype  : {test_data.dtype}')

## 5. Goruntuyu Gorsellestir

In [ ]:
def prepare_image(data):
    if data.ndim == 2:
        return np.stack([data, data, data], axis=0)
    elif data.ndim == 3 and data.shape[0] == 3:
        return data
    else:
        return np.stack([data[0], data[0], data[0]], axis=0)

rgb_data = prepare_image(test_data)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('cloudynight - FITS Goruntu 000 (BULUTLU)', fontsize=14, fontweight='bold')

for i, (ax, name, cmap) in enumerate(zip(axes,
    ['Kirmizi (R)', 'Yesil (G)', 'Mavi (B)'],
    ['Reds_r', 'Greens_r', 'Blues_r'])):
    layer = rgb_data[i]
    m, s  = layer.mean(), layer.std()
    ax.imshow(layer, cmap=cmap, vmin=m-s, vmax=m+s, origin='lower')
    ax.set_title(name)
    ax.axis('off')

plt.tight_layout()
plt.savefig('../outputs/goruntu_kanallar.png', dpi=100, bbox_inches='tight')
plt.show()
print('Kaydedildi: ../outputs/goruntu_kanallar.png')

## 6. HOG Ozellik Vektoru Cikarma

In [ ]:
def extract_hog_vectors(data, mask_coords):
    rgb     = prepare_image(data)
    rgb_hwc = ima.array2rgb(rgb)
    gray    = ima.rgb2gray(rgb_hwc)
    small   = ima.resize(gray, '25%')

    vectors = {}
    for direction, coordinates in mask_coords.items():
        try:
            the_mask = pmask.altaz(small.shape, coordinates[0], coordinates[1], rev=True)
            masked   = pmask.apply(small, the_mask)
            windowed = ima.find_window(masked)
            if windowed is None or windowed.size == 0:
                continue
            vec, hog_img = ima.hog(windowed, mchannel=False)
            if vec is not None:
                vectors[direction] = vec
        except Exception as e:
            print(f'  {direction} hatasi: {e}')
    return vectors

print('Test goruntusu icin HOG vektorleri cikariliyor...')
test_vectors = extract_hog_vectors(test_data, mask_coordinates)

print(f'Basarili yonler: {list(test_vectors.keys())}')
for d, v in test_vectors.items():
    print(f'  {d}: {v.shape[0]} boyutlu vektor')

## 7. Tum Goruntuleri Isle

In [ ]:
image_files = sorted([f for f in os.listdir(IMAGES_DIR) if f.endswith('.fits.bz2')])
direction_data = {d: {'X': [], 'y': []} for d in mask_coordinates}

print(f'{len(image_files)} goruntu isleniyor...')

for img_file in image_files:
    img_id = img_file.replace('.fits.bz2', '')
    label  = labels.get(img_id)
    if label is None:
        continue
    try:
        data    = read_fits_bz2(os.path.join(IMAGES_DIR, img_file))
        vectors = extract_hog_vectors(data, mask_coordinates)
        for direction, vec in vectors.items():
            direction_data[direction]['X'].append(vec)
            direction_data[direction]['y'].append(label)
        status = 'BULUTLU' if label == 0 else 'ACIK'
        print(f'  Goruntu {img_id}: {status} | {len(vectors)} yon')
    except Exception as e:
        print(f'  Goruntu {img_id} HATA: {e}')

print('\nYon basina veri sayisi:')
for d, dd in direction_data.items():
    print(f'  {d}: {len(dd["X"])} goruntu')

## 8. ML Modelleri Egit

In [ ]:
from sklearn.metrics import accuracy_score
import pandas as pd

print('ML Modelleri Egitiliyor...')
print('=' * 60)

results = []

for direction, data in direction_data.items():
    X_list, y_list = data['X'], data['y']
    if len(X_list) < 4 or len(np.unique(y_list)) < 2:
        continue

    X_flat  = [v.flatten() for v in X_list]
    min_len = min(len(v) for v in X_flat)
    X       = np.array([v[:min_len] for v in X_flat])
    y       = np.array(y_list)

    combined    = svm_clf.class_combiner(svm_clf.class_adder(X[y==1], 1), svm_clf.class_adder(X[y==0], 0))
    combined    = svm_clf.shuffle(combined)
    train, test = svm_clf.tts(combined, test_size=0.30)

    X_train, y_train = train[:, :-1], train[:, -1]
    X_test,  y_test  = test[:, :-1],  test[:, -1]

    if len(X_test) == 0:
        continue

    row = {'Yon': direction, 'Egitim': len(X_train), 'Test': len(X_test)}
    print(f'\n{direction} | Egitim: {len(X_train)} | Test: {len(X_test)}')

    for name, clf_obj, fit_fn in [
        ('SVM', svm_clf, lambda: svm_clf.classifier(X_train, y_train)),
        ('KNN', knn_clf, lambda: knn_clf.classifier(X_train, y_train, n_neighbors=3)),
        ('LR',  lr_clf,  lambda: lr_clf.classifier(X_train, y_train)),
        ('NB',  nb_clf,  lambda: nb_clf.classifier(X_train, y_train, tp='GAUSSIAN')),
    ]:
        try:
            clf    = fit_fn()
            y_pred = clf_obj.predict(clf, X_test)
            acc    = accuracy_score(y_test, y_pred)
            row[name] = round(acc, 4)
            print(f'  {name}: {acc:.4f}')
        except Exception as e:
            row[name] = None
            print(f'  {name}: HATA - {e}')

    results.append(row)

print('\n' + '=' * 60)
print('Egitim tamamlandi!')

## 9. Sonuc Tablosu ve Grafik

In [ ]:
if results:
    df = pd.DataFrame(results).set_index('Yon')
    model_cols = [c for c in ['SVM','KNN','LR','NB'] if c in df.columns]
    df_acc = df[model_cols].apply(pd.to_numeric, errors='coerce')

    print('DOGRULUK TABLOSU')
    print('=' * 50)
    print(df_acc.to_string())
    print('=' * 50)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('cloudynight Dataset - Model Karsilastirmasi (20 FITS goruntu)', fontsize=13, fontweight='bold')

    df_acc.plot(kind='bar', ax=axes[0], colormap='viridis', edgecolor='white')
    axes[0].set_title('Yone Gore Dogruluk')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_ylim(0, 1.1)
    axes[0].axhline(0.5, color='red', ls='--', alpha=0.5, label='Rastgele (%50)')
    axes[0].legend(loc='lower right')
    axes[0].tick_params(axis='x', rotation=0)

    means  = df_acc.mean()
    colors = ['#4C72B0','#DD8452','#55A868','#C44E52']
    bars   = axes[1].bar(means.index, means.values, color=colors[:len(means)], edgecolor='white', linewidth=1.5)
    axes[1].set_title('Ortalama Dogruluk (Tum Yonler)')
    axes[1].set_ylim(0, 1.1)
    axes[1].axhline(0.5, color='red', ls='--', alpha=0.5, label='Rastgele (%50)')
    for bar, val in zip(bars, means.values):
        if not np.isnan(val):
            axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02, f'{val:.3f}', ha='center', fontweight='bold')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('../outputs/model_karsilastirma.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Kaydedildi: ../outputs/model_karsilastirma.png')